# this work — Fase 5: Validação Human-in-the-Loop

**Interpretabilidade de Modelos de Linguagem em Português via Sparse Autoencoders**

---

**Objetivo:** Validar as 150 features selecionadas com protocolo híbrido LLM + humano.

**O que este notebook faz:**
1. Carrega modelo, SAE e features selecionadas (da Fase 3/4)
2. Extrai **top-20 exemplos ativadores** por feature dos corpora
3. Usa a **API Claude (Anthropic)** para gerar descrições automáticas de cada feature
4. Exporta um **template de validação** (JSON) para revisão manual
5. Após revisão manual, importa e computa **estatísticas de concordância**

**Protocolo de validação (por feature):**
1. LLM (Claude) gera descrição com base nos top-20 exemplos
2. Pesquisador examina 10-20 exemplos e avalia a descrição (escala 0-3)
3. Se discordar → registra interpretação própria
4. Classificação final = julgamento humano

**Pré-requisitos:**
- GPU com ≥16GB VRAM (T4 no Colab)
- Checkpoints da Fase 3 no Google Drive
- Chave da API Anthropic (obtida em [console.anthropic.com](https://console.anthropic.com))
- Tempo estimado: ~40 min (extração de exemplos + geração de descrições)

In [2]:
!pip install sae-lens transformer-lens anthropic -q
print()
print("Instalação concluída. Reinicie o runtime antes de continuar:")
print("  Runtime → Restart session (ou Ctrl+M .)")
print("  Depois, pule esta célula e execute a partir da próxima.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 5.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 145.1/145.1 kB 15.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 290.9/290.9 kB 31.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 195.2/195.2 kB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 469.4/469.4 kB 43.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 739.7/739.7 kB 56.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 49.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 91.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 102.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [1]:
import torch
import numpy as np
import json
import os
import time
from tqdm.auto import tqdm

if torch.cuda.is_available():
    device = 'cuda'
    gpu_name = torch.cuda.get_device_name()
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU: {gpu_name} ({gpu_mem:.1f} GB)')
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    device = 'mps'
    print('Usando Apple Silicon (MPS)')
else:
    device = 'cpu'
    print('Sem GPU detectada.')

print(f'Device: {device}')
print(f'PyTorch: {torch.__version__}')

GPU: Tesla T4 (15.6 GB)
Device: cuda
PyTorch: 2.10.0+cu128


In [2]:
from huggingface_hub import login
login()

In [3]:
from sae_lens import SAE, HookedSAETransformer

LAYER = 13
SAE_RELEASE = "gemma-scope-2b-pt-res-canonical"
SAE_ID = f"layer_{LAYER}/width_16k/canonical"

print("Carregando Gemma 2 2B...")
model = HookedSAETransformer.from_pretrained_no_processing(
    "gemma-2-2b",
    device=device,
    dtype=torch.float16,
)
print(f"Modelo: {model.cfg.model_name} | Layers: {model.cfg.n_layers} | d_model: {model.cfg.d_model}")

print()
print(f"Carregando SAE: {SAE_ID}...")
sae, cfg_dict, sparsity = SAE.from_pretrained(
    release=SAE_RELEASE,
    sae_id=SAE_ID,
    device=device,
)

HOOK_NAME = f"blocks.{LAYER}.hook_resid_post"
print(f"SAE: {sae.cfg.d_sae} features | d_in: {sae.cfg.d_in} | hook: {HOOK_NAME}")

Carregando Gemma 2 2B...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/818 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

Loaded pretrained model gemma-2-2b into HookedTransformer
Modelo: gemma-2-2b | Layers: 26 | d_model: 2304

Carregando SAE: layer_13/width_16k/canonical...


layer_13/width_16k/average_l0_84/params.(…):   0%|          | 0.00/302M [00:00<?, ?B/s]

SAE: 16384 features | d_in: 2304 | hook: blocks.13.hook_resid_post


/tmp/ipykernel_9746/3908203106.py:17: DeprecationWarning: Unpacking SAE objects is deprecated. SAE.from_pretrained() now returns only the SAE object. Use SAE.from_pretrained_with_cfg_and_sparsity() to get the config dict and sparsity as well.
  sae, cfg_dict, sparsity = SAE.from_pretrained(


In [4]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

Mounted at /content/drive


In [5]:
from google.colab import drive

SAVE_DIR = "./checkpoints/"  # adjust to your preferred path (e.g., ./checkpoints/ for Colab)
os.makedirs(SAVE_DIR, exist_ok=True)

MIN_ACTS = 100
LSI_THRESHOLD = 0.3
N_SELECT = 50

fase3_path = SAVE_DIR + "fase3_results.pt"
if os.path.exists(fase3_path):
    print(f"Carregando fase3_results.pt...")
    fase3 = torch.load(fase3_path, map_location="cpu")
    selected_pt = fase3["selected"]["pt"]
    selected_cross = fase3["selected"]["cross"]
    selected_en = fase3["selected"]["en"]
    all_selected = fase3["selected"]["all"]
    lsi_combined = fase3["lsi"]["combined"]
    lsi_wiki = fase3["lsi"]["wiki"]
    lsi_web = fase3["lsi"]["web"]
    pt_both = fase3["masks"]["pt_both"]
    has_examples = "examples" in fase3 and fase3["examples"]["pt"]
    if has_examples:
        print("  Exemplos da Fase 3 encontrados! Pulando extração.")
        examples_pt = fase3["examples"]["pt"]
        examples_cross = fase3["examples"]["cross"]
        examples_en = fase3["examples"]["en"]
else:
    print("Recomputando seleção a partir dos checkpoints individuais...")
    has_examples = False

    ckpt_files = ["stats_wiki_pt.pt", "stats_wiki_en.pt", "stats_mc4_pt.pt", "stats_c4_en.pt"]
    stats = {}
    for fname in ckpt_files:
        path = SAVE_DIR + fname
        if not os.path.exists(path):
            path = fname
        key = fname.replace("stats_", "").replace(".pt", "")
        stats[key] = torch.load(path, map_location="cpu")
        print(f"  Carregado: {fname}")

    freq_wiki_pt = stats["wiki_pt"]["counts"] / stats["wiki_pt"]["total_tokens"]
    freq_wiki_en = stats["wiki_en"]["counts"] / stats["wiki_en"]["total_tokens"]
    freq_mc4_pt = stats["mc4_pt"]["counts"] / stats["mc4_pt"]["total_tokens"]
    freq_c4_en = stats["c4_en"]["counts"] / stats["c4_en"]["total_tokens"]

    lsi_wiki = (freq_wiki_pt - freq_wiki_en) / (freq_wiki_pt + freq_wiki_en + 1e-10)
    lsi_web = (freq_mc4_pt - freq_c4_en) / (freq_mc4_pt + freq_c4_en + 1e-10)
    lsi_combined = (lsi_wiki + lsi_web) / 2

    total_counts = stats["wiki_pt"]["counts"] + stats["wiki_en"]["counts"] + stats["mc4_pt"]["counts"] + stats["c4_en"]["counts"]
    active = total_counts >= MIN_ACTS

    pt_wiki_mask = (lsi_wiki > LSI_THRESHOLD) & active
    pt_web_mask = (lsi_web > LSI_THRESHOLD) & active
    pt_both = pt_wiki_mask & pt_web_mask

    lsi_pt_rank = lsi_combined.clone()
    lsi_pt_rank[~active] = -2
    _, top_pt_idx = lsi_pt_rank.topk(N_SELECT)
    selected_pt = top_pt_idx.tolist()

    lsi_en_rank = lsi_combined.clone()
    lsi_en_rank[~active] = 2
    _, top_en_idx = lsi_en_rank.topk(N_SELECT, largest=False)
    selected_en = top_en_idx.tolist()

    total_freq = freq_wiki_pt + freq_wiki_en + freq_mc4_pt + freq_c4_en
    cross_score = lsi_combined.abs().clone()
    cross_score[~active] = 999
    min_cross_freq = total_freq[active].median()
    cross_score[total_freq < min_cross_freq] = 999
    _, top_cross_idx = cross_score.topk(N_SELECT, largest=False)
    selected_cross = top_cross_idx.tolist()

    all_selected = selected_pt + selected_cross + selected_en

print(f"\nFeatures: {len(selected_pt)} PT + {len(selected_cross)} cross + {len(selected_en)} EN = {len(all_selected)}")
print(f"Exemplos pré-carregados: {'Sim' if has_examples else 'Não (serão extraídos)'}")

Recomputando seleção a partir dos checkpoints individuais...
  Carregado: stats_wiki_pt.pt
  Carregado: stats_wiki_en.pt
  Carregado: stats_mc4_pt.pt
  Carregado: stats_c4_en.pt

Features: 50 PT + 50 cross + 50 EN = 150
Exemplos pré-carregados: Não (serão extraídos)


---
# Extração de Exemplos Ativadores

Para cada uma das 150 features, extraímos os **top-20 textos** que mais ativam a feature.
- Features PT e cross-lingual → exemplos de **Wikipedia PT**
- Features EN → exemplos de **Wikipedia EN**

Esses exemplos servem como base para a descrição automática (LLM) e para a validação manual.

In [6]:
if not has_examples:
    from datasets import load_dataset

    tokenizer = model.tokenizer
    SEQ_LEN = 256
    N_TOKENS_EXAMPLES = 2_000_000

    def collect_tokens(dataset, n_tokens, text_field="text", desc="Tokenizando"):
        all_ids = []
        n_articles = 0
        for article in tqdm(dataset, desc=desc):
            text = article[text_field]
            if not text or len(text) < 50:
                continue
            ids = tokenizer.encode(text, add_special_tokens=False)
            all_ids.extend(ids)
            n_articles += 1
            if len(all_ids) >= n_tokens:
                break
        all_ids = all_ids[:n_tokens]
        n_seqs = len(all_ids) // SEQ_LEN
        tokens = torch.tensor(all_ids[:n_seqs * SEQ_LEN], dtype=torch.long).reshape(n_seqs, SEQ_LEN)
        print(f"  {n_articles} textos -> {tokens.numel():,} tokens ({tokens.shape[0]} seqs)")
        return tokens

    print("Tokenizando Wikipedia PT (para exemplos)...")
    wiki_pt = load_dataset("wikimedia/wikipedia", "20231101.pt", split="train", streaming=True)
    tokens_wiki_pt = collect_tokens(wiki_pt, N_TOKENS_EXAMPLES, desc="Wiki PT")

    print("\nTokenizando Wikipedia EN (para exemplos)...")
    wiki_en = load_dataset("wikimedia/wikipedia", "20231101.en", split="train", streaming=True)
    tokens_wiki_en = collect_tokens(wiki_en, N_TOKENS_EXAMPLES, desc="Wiki EN")

    print("\nTokenização concluída!")
else:
    print("Exemplos já carregados da Fase 3. Pulando tokenização.")

Tokenizando Wikipedia PT (para exemplos)...


README.md: 0.00B [00:00, ?B/s]

Wiki PT: 0it [00:00, ?it/s]

  395 textos -> 1,999,872 tokens (7812 seqs)

Tokenizando Wikipedia EN (para exemplos)...


Resolving data files:   0%|          | 0/41 [00:00<?, ?it/s]

Wiki EN: 0it [00:00, ?it/s]

  315 textos -> 1,999,872 tokens (7812 seqs)

Tokenização concluída!


In [7]:
if not has_examples:
    BATCH_SIZE = 8
    N_EXAMPLES = 20

    @torch.no_grad()
    def find_top_examples(model, sae, tokens, feature_indices,
                          n_examples=N_EXAMPLES, batch_size=BATCH_SIZE, max_batches=500):
        hook = HOOK_NAME
        feat_list = list(feature_indices)
        results = {f: [] for f in feat_list}
        tok = model.tokenizer

        actual_batches = min(max_batches, (len(tokens) + batch_size - 1) // batch_size)
        for i in tqdm(range(0, actual_batches * batch_size, batch_size),
                      desc="Buscando exemplos", total=actual_batches):
            if i >= len(tokens):
                break
            batch = tokens[i:i+batch_size].to(device)
            _, cache = model.run_with_cache(batch, names_filter=lambda n: n == hook)
            acts = cache[hook]
            feat_acts = sae.encode(acts)

            for f_idx in feat_list:
                vals = feat_acts[:, :, f_idx]
                for b in range(batch.shape[0]):
                    max_val, max_pos = vals[b].max(dim=0)
                    if max_val.item() > 0:
                        pos = max_pos.item()
                        start = max(0, pos - 10)
                        end = min(batch.shape[1], pos + 15)
                        ctx = tok.decode(batch[b, start:end].tolist())
                        results[f_idx].append({"act": max_val.item(), "context": ctx})

            del cache, acts, feat_acts
            if device == "cuda":
                torch.cuda.empty_cache()

        for f in results:
            results[f].sort(key=lambda x: x["act"], reverse=True)
            results[f] = results[f][:n_examples]
        return results

    print("Extraindo exemplos para features PT-específicas (Wiki PT)...")
    examples_pt = find_top_examples(model, sae, tokens_wiki_pt, selected_pt)

    print("\nExtraindo exemplos para features cross-lingual (Wiki PT)...")
    examples_cross = find_top_examples(model, sae, tokens_wiki_pt, selected_cross)

    print("\nExtraindo exemplos para features EN-específicas (Wiki EN)...")
    examples_en = find_top_examples(model, sae, tokens_wiki_en, selected_en)

    examples_ckpt = {
        "pt": examples_pt,
        "cross": examples_cross,
        "en": examples_en,
    }
    torch.save(examples_ckpt, SAVE_DIR + "fase5_examples.pt")
    print(f"\nExemplos salvos em: {SAVE_DIR}fase5_examples.pt")
else:
    print("Exemplos já disponíveis. Pulando extração.")

print(f"\nExemplos extraídos:")
print(f"  PT: {sum(len(v) for v in examples_pt.values())} exemplos para {len(examples_pt)} features")
print(f"  Cross: {sum(len(v) for v in examples_cross.values())} exemplos para {len(examples_cross)} features")
print(f"  EN: {sum(len(v) for v in examples_en.values())} exemplos para {len(examples_en)} features")

Extraindo exemplos para features PT-específicas (Wiki PT)...


Buscando exemplos:   0%|          | 0/500 [00:00<?, ?it/s]


Extraindo exemplos para features cross-lingual (Wiki PT)...


Buscando exemplos:   0%|          | 0/500 [00:00<?, ?it/s]


Extraindo exemplos para features EN-específicas (Wiki EN)...


Buscando exemplos:   0%|          | 0/500 [00:00<?, ?it/s]


Exemplos salvos em: ./checkpoints/fase5_examples.pt

Exemplos extraídos:
  PT: 1000 exemplos para 50 features
  Cross: 1000 exemplos para 50 features
  EN: 843 exemplos para 50 features


---
# Geração de Descrições via LLM (Claude/Anthropic)

Para cada feature, enviamos os top-20 exemplos ativadores ao **Claude (Anthropic)** e pedimos uma descrição concisa do que a feature detecta.

**Obter chave API:**
1. Acesse [console.anthropic.com](https://console.anthropic.com)
2. Crie uma API key nas configurações da conta
3. Cole a chave na célula abaixo

A descrição gerada será depois avaliada pelo pesquisador (escala 0-3).

In [8]:
import anthropic
from google.colab import userdata

try:
    ANTHROPIC_API_KEY = userdata.get('ANTHROPIC_API_KEY')
    print("Chave API carregada dos secrets do Colab.")
except Exception:
    ANTHROPIC_API_KEY = input("Cole sua chave API da Anthropic: ").strip()

client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

test = client.messages.create(
    model="claude-opus-4-20250514",
    max_tokens=50,
    messages=[{"role": "user", "content": "Responda apenas 'OK' se estiver funcionando."}],
)
print(f"Teste Claude: {test.content[0].text.strip()}")
print("API configurada com sucesso!")

Chave API carregada dos secrets do Colab.
Teste Claude: OK
API configurada com sucesso!


In [9]:
def get_category(feat_id):
    if feat_id in selected_pt:
        return "PT-specific"
    elif feat_id in selected_en:
        return "EN-specific"
    else:
        return "Cross-lingual"

def get_examples(feat_id):
    if feat_id in examples_pt:
        return examples_pt[feat_id]
    elif feat_id in examples_cross:
        return examples_cross[feat_id]
    elif feat_id in examples_en:
        return examples_en[feat_id]
    return []

def build_prompt(feat_id, examples, lsi_val, category):
    examples_text = ""
    for i, ex in enumerate(examples[:20]):
        examples_text += '{0}. [activation: {1:.2f}] "{2}"\n'.format(i+1, ex["act"], ex["context"])

    prompt = (
        "You are analyzing features extracted from a Sparse Autoencoder (SAE) "
        "applied to the Gemma 2 2B language model at layer 13 (residual stream).\n\n"
        "Below are the top activating text segments for a specific feature. Each segment "
        "shows the text context around the token that most strongly activates this feature, "
        "along with the activation strength.\n\n"
        "Feature ID: {feat_id}\n"
        "Language Specificity Index (LSI): {lsi_val:+.4f} "
        "(range: -1.0 = English-only, +1.0 = Portuguese-only, 0.0 = cross-lingual)\n"
        "Category: {category}\n\n"
        "Top activating examples (sorted by activation strength):\n"
        "{examples_text}\n"
        "Based on these examples, provide:\n"
        "1. A concise description (1-2 sentences) of what this feature detects or represents.\n"
        "2. If applicable, identify the specific linguistic phenomenon "
        "(e.g., grammatical gender, crase, clitic pronouns, verb conjugation, prepositions, contractions, etc.).\n"
        "3. Rate your confidence: HIGH (clear pattern), MEDIUM (likely pattern), or LOW (unclear).\n\n"
        "Respond in this exact format:\n"
        "DESCRIPTION: [your description]\n"
        "PHENOMENON: [linguistic phenomenon or 'general' if not specific]\n"
        "CONFIDENCE: [HIGH/MEDIUM/LOW]"
    ).format(feat_id=feat_id, lsi_val=lsi_val, category=category, examples_text=examples_text)
    return prompt

ckpt_path = SAVE_DIR + "fase5_descriptions.json"
if os.path.exists(ckpt_path):
    with open(ckpt_path, "r") as f:
        descriptions = json.load(f)
    print(f"Carregadas {len(descriptions)} descrições do checkpoint.")
    remaining = [f for f in all_selected if str(f) not in descriptions]
else:
    descriptions = {}
    remaining = list(all_selected)

print(f"Features a processar: {len(remaining)} (de {len(all_selected)} total)")

errors = []
for i, feat_id in enumerate(remaining):
    examples = get_examples(feat_id)
    if not examples:
        descriptions[str(feat_id)] = {
            "description": "No examples available",
            "phenomenon": "unknown",
            "confidence": "LOW",
            "raw": "",
        }
        continue

    lsi_val = lsi_combined[feat_id].item()
    category = get_category(feat_id)
    prompt = build_prompt(feat_id, examples, lsi_val, category)

    try:
        response = client.messages.create(
              model="claude-opus-4-20250514",
              max_tokens=300,
              messages=[{"role": "user", "content": prompt}],
          )
        text = response.content[0].text.strip()
        desc = ""
        phenom = ""
        conf = ""
        for line in text.split("\n"):
            line = line.strip()
            if line.startswith("DESCRIPTION:"):
                desc = line[len("DESCRIPTION:"):].strip()
            elif line.startswith("PHENOMENON:"):
                phenom = line[len("PHENOMENON:"):].strip()
            elif line.startswith("CONFIDENCE:"):
                conf = line[len("CONFIDENCE:"):].strip()

        descriptions[str(feat_id)] = {
            "description": desc or text,
            "phenomenon": phenom or "unknown",
            "confidence": conf or "MEDIUM",
            "raw": text,
        }

        if (i + 1) % 10 == 0:
            with open(ckpt_path, "w") as f:
                json.dump(descriptions, f, ensure_ascii=False, indent=2)
            print(f"  [{i+1}/{len(remaining)}] Checkpoint salvo")

    except Exception as e:
        errors.append((feat_id, str(e)))
        print(f"  Erro na feature {feat_id}: {e}")

    time.sleep(1.5)

with open(ckpt_path, "w") as f:
    json.dump(descriptions, f, ensure_ascii=False, indent=2)

print(f"\nDescrições geradas: {len(descriptions)}")
if errors:
    print(f"Erros: {len(errors)}")
    for fid, err in errors[:5]:
        print(f"  Feature {fid}: {err}")
print(f"Salvo em: {ckpt_path}")

Carregadas 0 descrições do checkpoint.
Features a processar: 150 (de 150 total)
  [10/150] Checkpoint salvo
  [20/150] Checkpoint salvo
  [30/150] Checkpoint salvo
  [40/150] Checkpoint salvo
  [50/150] Checkpoint salvo
  [60/150] Checkpoint salvo
  [70/150] Checkpoint salvo
  [80/150] Checkpoint salvo
  [90/150] Checkpoint salvo
  [100/150] Checkpoint salvo
  [110/150] Checkpoint salvo
  [120/150] Checkpoint salvo
  [130/150] Checkpoint salvo
  [140/150] Checkpoint salvo
  [150/150] Checkpoint salvo

Descrições geradas: 150
Salvo em: ./checkpoints/fase5_descriptions.json


In [10]:
print("=" * 90)
print("DESCRIÇÕES GERADAS — AMOSTRA (top-10 PT-específicas)")
print("=" * 90)

for rank, feat_id in enumerate(selected_pt[:10]):
    desc_data = descriptions.get(str(feat_id), {})
    lsi_val = lsi_combined[feat_id].item()
    examples = get_examples(feat_id)

    print(f"\n{'━' * 90}")
    print(f"Feature {feat_id} | LSI={lsi_val:+.4f} | #{rank+1} PT | {desc_data.get('confidence', '?')}")
    print(f"━" * 90)
    print(f"  LLM: {desc_data.get('description', 'N/A')}")
    print(f"  Fenômeno: {desc_data.get('phenomenon', 'N/A')}")
    print(f"  Top-5 exemplos:")
    for ex in examples[:5]:
        print(f'    [{ex["act"]:.1f}] ...{ex["context"][:80]}...')

DESCRIÇÕES GERADAS — AMOSTRA (top-10 PT-específicas)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Feature 10496 | LSI=+0.9982 | #1 PT | HIGH
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  LLM: This feature activates strongly on Portuguese text containing the word "é" (third person singular present tense of the verb "ser" - to be), particularly when it appears in descriptive or definitional contexts.
  Fenômeno: Portuguese copula verb "ser" (3rd person singular present)
  Top-5 exemplos:
    [21.4] ... Aires).

 Economia 

A economia da Argentina é a terceira maior da América Lati...
    [20.8] ...os. Nas regiões do sul, os verões são mornos e invernos frios com fortes nevoada...
    [20.6] ...

Etimologia 
Em grego antigo, Atenas era chamada  (Athénai), em homenagem à deu...
    [19.9] ... 156 km de Belo Horizonte. É considerada polo para algumas cidades de pequeno po...
    [19.8] ...ises raci

---
# Protocolo de Validação Manual

A célula seguinte gera um **template de validação** (JSON) salvo no Google Drive.

**Para cada feature, você deve preencher:**

| Campo | Valores | Descrição |
|---|---|---|
| `human_score` | 0-3 | 0=irrelevante, 1=parcial, 2=boa, 3=perfeita |
| `human_description` | texto | Sua interpretação (obrigatório se score ≤ 1) |
| `human_notes` | texto | Observações opcionais |

**Escala de concordância:**
- **3 — Perfeita:** A descrição do LLM captura exatamente o que a feature faz
- **2 — Boa:** A descrição está correta mas incompleta ou imprecisa
- **1 — Parcial:** A descrição captura algo, mas perde o padrão principal
- **0 — Irrelevante:** A descrição está errada ou a feature não é interpretável

**Dica:** Comece pelas 50 PT-específicas (mais importantes para o this work). As cross-lingual e EN são controles.

In [11]:
validation_template = []

for feat_id in all_selected:
    desc_data = descriptions.get(str(feat_id), {})
    examples = get_examples(feat_id)
    lsi_val = lsi_combined[feat_id].item()
    category = get_category(feat_id)

    entry = {
        "feature_id": feat_id,
        "lsi": round(lsi_val, 4),
        "category": category,
        "llm_description": desc_data.get("description", ""),
        "llm_phenomenon": desc_data.get("phenomenon", ""),
        "llm_confidence": desc_data.get("confidence", ""),
        "top_examples": [
            {"activation": round(ex["act"], 2), "context": ex["context"]}
            for ex in examples[:20]
        ],
        "human_score": None,
        "human_description": None,
        "human_notes": None,
    }
    validation_template.append(entry)

template_path = SAVE_DIR + "fase5_validation_template.json"
with open(template_path, "w", encoding="utf-8") as f:
    json.dump(validation_template, f, ensure_ascii=False, indent=2)

print(f"Template de validação salvo em: {template_path}")
print(f"Total de features: {len(validation_template)}")
print(f"  PT-específicas: {sum(1 for e in validation_template if e['category'] == 'PT-specific')}")
print(f"  Cross-lingual:  {sum(1 for e in validation_template if e['category'] == 'Cross-lingual')}")
print(f"  EN-específicas: {sum(1 for e in validation_template if e['category'] == 'EN-specific')}")
print()
print("INSTRUÇÕES:")
print("  1. Abra o arquivo JSON no Drive (ou baixe para editar)")
print("  2. Para cada feature, preencha: human_score (0-3)")
print("  3. Se score ≤ 1, preencha human_description com sua interpretação")
print("  4. Após revisar, execute a próxima célula para importar e computar estatísticas")

Template de validação salvo em: ./checkpoints/fase5_validation_template.json
Total de features: 150
  PT-específicas: 50
  Cross-lingual:  50
  EN-específicas: 50

INSTRUÇÕES:
  1. Abra o arquivo JSON no Drive (ou baixe para editar)
  2. Para cada feature, preencha: human_score (0-3)
  3. Se score ≤ 1, preencha human_description com sua interpretação
  4. Após revisar, execute a próxima célula para importar e computar estatísticas


---
# Importação e Análise da Validação

Após revisar e preencher o template JSON, execute as células abaixo para:
1. Importar as respostas
2. Computar estatísticas de concordância
3. Gerar resumo para o paper

**Execute as células abaixo SOMENTE após concluir a validação manual.**

In [13]:
template_path = SAVE_DIR + "fase5_validation_template.json"

with open(template_path, "r", encoding="utf-8") as f:
    validated = json.load(f)

scored = [e for e in validated if e["human_score"] is not None]
unscored = [e for e in validated if e["human_score"] is None]

print("=" * 70)
print("ESTATÍSTICAS DE VALIDAÇÃO")
print("=" * 70)
print(f"Features avaliadas:    {len(scored)}/{len(validated)}")
print(f"Features pendentes:    {len(unscored)}")

if not scored:
    print("\nNenhuma feature avaliada ainda. Preencha o template e execute novamente.")
else:
    scores = [e["human_score"] for e in scored]
    mean_score = np.mean(scores)
    print(f"\nScore médio:           {mean_score:.2f}")
    print(f"Distribuição de scores:")
    for s in range(4):
        count = scores.count(s)
        pct = count / len(scores) * 100
        bar = "█" * int(pct / 2)
        print(f"  {s} {'(irrelevante)' if s==0 else '(parcial)' if s==1 else '(boa)' if s==2 else '(perfeita)':>14s}: {count:3d} ({pct:5.1f}%) {bar}")

    agreement = sum(1 for s in scores if s >= 2) / len(scores)
    print(f"\nConcordância (score ≥ 2): {agreement:.1%}")

    for cat in ["PT-specific", "Cross-lingual", "EN-specific"]:
        cat_scores = [e["human_score"] for e in scored if e["category"] == cat]
        if cat_scores:
            cat_mean = np.mean(cat_scores)
            cat_agree = sum(1 for s in cat_scores if s >= 2) / len(cat_scores)
            print(f"\n  {cat}:")
            print(f"    Avaliadas: {len(cat_scores)}")
            print(f"    Score médio: {cat_mean:.2f}")
            print(f"    Concordância: {cat_agree:.1%}")

    human_descs = [e for e in scored if e.get("human_description")]
    if human_descs:
        print(f"\nDescrições humanas alternativas: {len(human_descs)}")
        disagreements = [e for e in scored if e["human_score"] <= 1 and e.get("human_description")]
        print(f"Desacordos com descrição própria: {len(disagreements)}")

    confidence_scores = {}
    for e in scored:
        conf = descriptions.get(str(e["feature_id"]), {}).get("confidence", "UNKNOWN")
        if conf not in confidence_scores:
            confidence_scores[conf] = []
        confidence_scores[conf].append(e["human_score"])

    print(f"\nScore médio por confiança do LLM:")
    for conf in ["HIGH", "MEDIUM", "LOW"]:
        if conf in confidence_scores:
            cs = confidence_scores[conf]
            print(f"  {conf}: {np.mean(cs):.2f} (n={len(cs)})")

ESTATÍSTICAS DE VALIDAÇÃO
Features avaliadas:    150/150
Features pendentes:    0

Score médio:           1.91
Distribuição de scores:
  0  (irrelevante):   2 (  1.3%) 
  1      (parcial):  60 ( 40.0%) ████████████████████
  2          (boa):  38 ( 25.3%) ████████████
  3     (perfeita):  50 ( 33.3%) ████████████████

Concordância (score ≥ 2): 58.7%

  PT-specific:
    Avaliadas: 50
    Score médio: 1.62
    Concordância: 44.0%

  Cross-lingual:
    Avaliadas: 50
    Score médio: 2.20
    Concordância: 70.0%

  EN-specific:
    Avaliadas: 50
    Score médio: 1.90
    Concordância: 62.0%

Descrições humanas alternativas: 18
Desacordos com descrição própria: 12

Score médio por confiança do LLM:
  HIGH: 2.06 (n=123)
  MEDIUM: 1.16 (n=25)
  LOW: 2.00 (n=2)


In [14]:
if scored:
    print("=" * 90)
    print("CATÁLOGO DE FEATURES VALIDADAS")
    print("=" * 90)

    fmt = "{:<8} {:<10} {:<14} {:<6} {:<50}"
    print(fmt.format("Feature", "LSI", "Categoria", "Score", "Descrição Final"))
    print("-" * 90)

    for e in sorted(scored, key=lambda x: (-x["lsi"] if x["category"] == "PT-specific" else x["lsi"])):
        final_desc = e.get("human_description") or e["llm_description"]
        final_desc = final_desc[:48] + ".." if len(final_desc) > 48 else final_desc
        print(fmt.format(
            e["feature_id"],
            f'{e["lsi"]:+.4f}',
            e["category"],
            e["human_score"],
            final_desc,
        ))

CATÁLOGO DE FEATURES VALIDADAS
Feature  LSI        Categoria      Score  Descrição Final                                   
------------------------------------------------------------------------------------------
11718    -1.0000    EN-specific    2      This feature activates on numerical percentages ..
10496    +0.9982    PT-specific    2      General Portuguese text detector, broadly activa..
1906     +0.9981    PT-specific    2      This feature detects Portuguese text segments, p..
8718     +0.9977    PT-specific    2      Portuguese encyclopedic/definitional text, also ..
11289    -0.9973    EN-specific    1      This feature appears to activate on descriptive ..
10533    -0.9968    EN-specific    2      This feature appears to detect the word "Creole"..
16057    +0.9965    PT-specific    3      This feature activates strongly on the Portugues..
10531    -0.9962    EN-specific    1      This feature appears to detect explanatory or pr..
4584     +0.9953    PT-specific    3     

In [15]:
results_fase5 = {
    "validation": validated,
    "descriptions": descriptions,
    "statistics": {
        "total": len(validated),
        "scored": len(scored),
        "mean_score": float(np.mean([e["human_score"] for e in scored])) if scored else None,
        "agreement_rate": float(sum(1 for e in scored if e["human_score"] >= 2) / len(scored)) if scored else None,
    },
    "selected_pt": selected_pt,
    "selected_cross": selected_cross,
    "selected_en": selected_en,
}

save_path = SAVE_DIR + "fase5_validation_results.pt"
torch.save(results_fase5, save_path)
print(f"Resultados salvos em: {save_path}")

print()
print("=" * 70)
print("ARQUIVOS GERADOS (todos no Drive)")
print("=" * 70)
print(f"  {SAVE_DIR}fase5_examples.pt              — exemplos ativadores")
print(f"  {SAVE_DIR}fase5_descriptions.json         — descrições do LLM")
print(f"  {SAVE_DIR}fase5_validation_template.json  — template para validação manual")
print(f"  {SAVE_DIR}fase5_validation_results.pt     — resultados finais")

Resultados salvos em: ./checkpoints/fase5_validation_results.pt

ARQUIVOS GERADOS (todos no Drive)
  ./checkpoints/fase5_examples.pt              — exemplos ativadores
  ./checkpoints/fase5_descriptions.json         — descrições do LLM
  ./checkpoints/fase5_validation_template.json  — template para validação manual
  ./checkpoints/fase5_validation_results.pt     — resultados finais

